# Comparison Only: All 7 Models Side by Side (Colab, Drive)

Reads whatever prediction caches already exist for each model (produced by `NVTV_All_Models_RunAndRetry_Colab.ipynb`, on Kelvin2, or a mix of both) and builds the coverage + comparison tables. **Loads no models and runs no inference** -- if a model has 0 cached predictions, it just shows up with zero coverage in the tables rather than erroring.

## 1. Configuration

In [ ]:
from __future__ import annotations

import json
import os
import re
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Optional

import pandas as pd

from google.colab import drive
drive.mount("/content/drive")


@dataclass
class Config:
    # Google Drive copy of the same NVTV_PublicData1 folder used on Kelvin2.
    project_root: Path = Path("/content/drive/MyDrive/NVTV_PublicData1")
    output_subdir: str = "automatic_ground_truth_v4"

    @property
    def artifact_dir(self) -> Path:
        return self.project_root / self.output_subdir

    @property
    def clip_dir(self) -> Path:
        return self.artifact_dir / "clips"

    @property
    def ground_truth_file(self) -> Path:
        return self.artifact_dir / "ground_truth_metadata_615.json"

    @property
    def qwen_output_dir(self) -> Path:
        return self.artifact_dir / "qwen_vl_comparison"


CFG = Config()

# All 7 models this compares -- present here regardless of which ones
# actually have cached predictions yet (see Section 3).
MODEL_CONFIGS = [
    {"name": "Qwen3-VL-2B-Instruct", "repo_id": "Qwen/Qwen3-VL-2B-Instruct"},
    {"name": "Qwen3-VL-2B-Thinking", "repo_id": "Qwen/Qwen3-VL-2B-Thinking"},
    {"name": "Qwen3-VL-4B-Instruct", "repo_id": "Qwen/Qwen3-VL-4B-Instruct"},
    {"name": "Qwen3-VL-4B-Thinking", "repo_id": "Qwen/Qwen3-VL-4B-Thinking"},
    {"name": "Qwen3-VL-8B-Instruct", "repo_id": "Qwen/Qwen3-VL-8B-Instruct", "quantize_int4": True},
    {"name": "InternVL3-2B", "repo_id": "OpenGVLab/InternVL3-2B-hf", "trust_remote_code": True},
    {"name": "InternVL3-8B", "repo_id": "OpenGVLab/InternVL3-8B-hf", "trust_remote_code": True, "quantize_int4": True},
]
MODEL_LOAD_FAILURES: dict[str, str] = {}  # always empty here -- nothing is loaded

RUN_TAG = "ALL_MODELS"
QWEN_PROMPT_VERSION = "nvtv-qwen3vl-family-v3"  # must match whatever produced the caches

# Same controlled vocabulary the automatic pipeline and the run/retry
# notebook use, so visual_tags accuracy is meaningful.
VISUAL_LABELS = (
    "news studio", "interview", "press conference", "public meeting",
    "panel discussion", "person speaking", "crowd of people", "protest",
    "indoor scene", "outdoor scene", "city street", "building exterior",
    "office", "stage", "podium", "fire engine", "emergency services",
    "police", "hospital", "school", "graphic or title card",
    "text on screen", "logo", "presentation slide", "landscape",
    "rural countryside", "vehicle", "road", "sign or banner",
    "audience", "reporter", "politician", "firefighter", "uniform",
)

print("Ground truth file:", CFG.ground_truth_file)
print("Reading predictions from:", CFG.qwen_output_dir)


Mounted at /content/drive
Ground truth file: /content/drive/MyDrive/NVTV_PublicData1/automatic_ground_truth_v4/ground_truth_metadata_615.json
Reading predictions from: /content/drive/MyDrive/NVTV_PublicData1/automatic_ground_truth_v4/qwen_vl_comparison


## 2. Text-normalisation and scoring helpers

In [ ]:
%pip install -q rouge-score==0.1.2

  Preparing metadata (setup.py) ... done


In [ ]:
from rouge_score import rouge_scorer


def normalize_space(value: Any) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()


def normalize_key(value: Any) -> str:
    return re.sub(r"[^a-z0-9]+", " ", normalize_space(value).lower()).strip()


def unique_strings(values: Iterable[str]) -> list[str]:
    output, seen = [], set()
    for value in values:
        clean = normalize_space(value)
        key = normalize_key(clean)
        if clean and key and key not in seen:
            seen.add(key)
            output.append(clean)
    return output


def plausible_ocr_text(value: Any) -> bool:
    text = normalize_space(value)
    if not text or len(text) > 160:
        return False
    non_space = [c for c in text if not c.isspace()]
    alphanumeric = sum(c.isalnum() for c in non_space)
    if not non_space or alphanumeric / len(non_space) < 0.50:
        return False
    key = normalize_key(text)
    return len(key) >= 2 or key.isdigit()


NUMBER_WORDS = {
    "zero": 0, "none": 0, "nobody": 0, "one": 1, "single": 1,
    "two": 2, "couple": 2, "three": 3, "few": 3, "four": 4,
    "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
}


def parse_people_count(value: Any) -> Optional[int]:
    text = normalize_key(value)
    match = re.search(r"\b\d+\b", text)
    if match:
        return int(match.group())
    for token in text.split():
        if token in NUMBER_WORDS:
            return NUMBER_WORDS[token]
    return None


# A single "rouge" score per field, deliberately not a rouge1/rouge2/rougeL
# breakdown -- see NVTV_All_Models_RunAndRetry_Colab.ipynb for why.
ROUGE_SCORER = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def rouge_recall(reference: Iterable[str], prediction: Iterable[str]) -> float:
    reference_text = normalize_space(" ".join(reference or []))
    prediction_text = normalize_space(" ".join(prediction or []))
    if not reference_text and not prediction_text:
        return 1.0
    if not reference_text or not prediction_text:
        return 0.0
    return float(ROUGE_SCORER.score(reference_text, prediction_text)["rougeL"].recall)


def multilabel_accuracy(
    reference: Iterable[str], prediction: Iterable[str], vocabulary: Iterable[str],
) -> float:
    reference_set = {normalize_key(v) for v in (reference or [])}
    prediction_set = {normalize_key(v) for v in (prediction or [])}
    vocabulary_keys = [normalize_key(v) for v in vocabulary]
    correct = sum((v in reference_set) == (v in prediction_set) for v in vocabulary_keys)
    return correct / len(vocabulary_keys)


def bounded(value: float) -> float:
    return float(min(1.0, max(0.0, value)))


def write_json_atomic(payload: Any, path: Path) -> None:
    temp_path = path.with_name(path.stem + ".partial.json")
    with open(temp_path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)
    os.replace(temp_path, path)


def write_csv_atomic(frame: pd.DataFrame, path: Path) -> None:
    temp_path = path.with_name(path.stem + ".partial.csv")
    frame.to_csv(temp_path, index=False)
    os.replace(temp_path, path)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


print("Helpers ready.")


Helpers ready.


## 3. Load ground truth and build the clip list

In [ ]:
if not CFG.ground_truth_file.exists():
    raise FileNotFoundError(
        f"Ground truth file not found: {CFG.ground_truth_file}\n"
        "Check Config.project_root above -- it should point at your Google "
        "Drive folder containing automatic_ground_truth_v4/."
    )

with open(CFG.ground_truth_file, encoding="utf-8") as handle:
    GROUND_TRUTH = json.load(handle)

ok_clips = [clip for clip in GROUND_TRUTH.get("clips", []) if clip.get("status") == "ok"]
print(f"Ground truth clips total: {len(GROUND_TRUTH.get('clips', []))}")
print(f"Usable ('ok' status) clips: {len(ok_clips)}")

CLIP_ROWS = []
for clip in ok_clips:
    clip_path = CFG.clip_dir / f"{clip['clip_id']}.mp4"
    if not clip_path.exists():
        print(f"WARNING: clip file missing on disk, skipping: {clip_path}")
        continue
    CLIP_ROWS.append({
        "clip_id": clip["clip_id"],
        "source_id": clip.get("source_id", ""),
        "source_video": clip.get("source_video", ""),
        "split": clip.get("split", ""),
        "clip_path": clip_path,
        "ground_truth_metadata": clip["ground_truth_metadata"],
    })

CLIP_MANIFEST = pd.DataFrame(CLIP_ROWS)
print(f"Clips with a video file on disk: {len(CLIP_MANIFEST)}")
CLIP_MANIFEST[["clip_id", "split", "source_video"]].head()


Ground truth clips total: 615
Usable ('ok' status) clips: 615
Clips with a video file on disk: 615


,clip_id,split,source_video
0,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,,10. Jacobin Launch 230316.mp4
1,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,,10. Jacobin Launch 230316.mp4
2,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,,10. Jacobin Launch 230316.mp4
3,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,,10. Jacobin Launch 230316.mp4
4,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,,10. Jacobin Launch 230316.mp4


## 4. Load every model's cached predictions

No model is loaded and no inference runs here -- this just reads whatever `predictions_<model>.json` files already exist in `CFG.qwen_output_dir`. A model with no cache yet, or a stale/mismatched `prompt_version`, comes back as an empty dict and simply shows zero coverage below rather than erroring.

In [ ]:
def predictions_file_for(model_name: str) -> Path:
    safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", model_name)
    return CFG.qwen_output_dir / f"predictions_{safe_name}.json"


def load_predictions_for(model_name: str) -> dict[str, dict]:
    path = predictions_file_for(model_name)
    if not path.exists():
        print(f"No cache found for {model_name}: {path}")
        return {}
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    if payload.get("prompt_version") != QWEN_PROMPT_VERSION:
        print(f"Prompt version mismatch for {model_name}; ignoring its cache.")
        return {}
    records = {
        record["clip_id"]: record
        for record in payload.get("records", [])
        if record.get("clip_id")
    }
    print(f"Loaded cache: {path.name} ({len(records)} records)")
    return records


ALL_QWEN_RECORDS: dict[str, dict[str, dict]] = {}
for model_cfg in MODEL_CONFIGS:
    ALL_QWEN_RECORDS[model_cfg["name"]] = load_predictions_for(model_cfg["name"])


Loaded cache: predictions_Qwen3-VL-2B-Instruct.json (615 records)
Loaded cache: predictions_Qwen3-VL-2B-Thinking.json (615 records)
Loaded cache: predictions_Qwen3-VL-4B-Instruct.json (615 records)
Loaded cache: predictions_Qwen3-VL-4B-Thinking.json (615 records)
Loaded cache: predictions_Qwen3-VL-8B-Instruct.json (615 records)
Loaded cache: predictions_InternVL3-2B.json (615 records)
Loaded cache: predictions_InternVL3-8B.json (615 records)


## 5. Score every model's output against the automatic ground truth

In [ ]:
SCORED_FIELDS = ("on_screen_text", "visual_tags", "keywords", "people_count", "transcript")


def ground_truth_value(clip_metadata: dict, field: str) -> Any:
    evidence = clip_metadata.get(field, {})
    return evidence.get("value") if isinstance(evidence, dict) else evidence


def ground_truth_agreement_tier(clip_metadata: dict, field: str) -> str:
    evidence = clip_metadata.get(field, {})
    if isinstance(evidence, dict):
        return evidence.get("agreement_tier", "not_scored")
    return "not_scored"


def score_clip_field(field: str, reference: Any, prediction: Any) -> dict[str, float]:
    """Returns {metric_name: value}; empty dict means "not scoreable", not zero."""
    if field in {"on_screen_text", "keywords"}:
        return {"rouge": rouge_recall(reference or [], prediction or [])}
    if field == "visual_tags":
        return {"accuracy": multilabel_accuracy(reference or [], prediction or [], VISUAL_LABELS)}
    if field == "people_count":
        expected = parse_people_count(reference)
        if expected is None:
            return {}
        return {"accuracy": float(prediction == expected)}
    if field == "transcript":
        reference_text = reference if isinstance(reference, str) else None
        return {"rouge": rouge_recall(
            [reference_text] if reference_text else [],
            [prediction] if prediction else [],
        )}
    raise ValueError(f"Unsupported field: {field}")


rows = []
for model_cfg in MODEL_CONFIGS:
    model_name = model_cfg["name"]
    records_for_model = ALL_QWEN_RECORDS.get(model_name, {})
    for row in CLIP_MANIFEST.itertuples(index=False):
        prediction_record = records_for_model.get(row.clip_id, {})
        if prediction_record.get("status") != "ok":
            continue
        qwen_metadata = prediction_record["metadata"]

        predictions = {
            "on_screen_text": qwen_metadata["on_screen_text"],
            "visual_tags": qwen_metadata["visual_tags"],
            "keywords": qwen_metadata["keywords"],
            "people_count": qwen_metadata["people_count_numeric"],
            "transcript": qwen_metadata["transcript_guess"],
        }
        for field in SCORED_FIELDS:
            reference = ground_truth_value(row.ground_truth_metadata, field)
            tier = ground_truth_agreement_tier(row.ground_truth_metadata, field)
            for metric_name, value in score_clip_field(field, reference, predictions[field]).items():
                rows.append({
                    "model": model_name,
                    "clip_id": row.clip_id,
                    "split": row.split,
                    "field": field,
                    "metric": metric_name,
                    "value": bounded(value),
                    "ground_truth_value": reference,
                    "qwen_value": predictions[field],
                    "ground_truth_agreement_tier": tier,
                })

QWEN_ACCURACY_REPORT = pd.DataFrame(rows)
QWEN_ACCURACY_REPORT_FILE = CFG.qwen_output_dir / f"qwen_vl_accuracy_report__{RUN_TAG}.csv"
write_csv_atomic(QWEN_ACCURACY_REPORT, QWEN_ACCURACY_REPORT_FILE)
print(f"Scored {len(QWEN_ACCURACY_REPORT)} (model, clip, field) rows.")
print("Saved to:", QWEN_ACCURACY_REPORT_FILE)
QWEN_ACCURACY_REPORT.head(10)


Scored 21140 (model, clip, field) rows.
Saved to: /content/drive/MyDrive/NVTV_PublicData1/automatic_ground_truth_v4/qwen_vl_comparison/qwen_vl_accuracy_report__ALL_MODELS.csv


,model,clip_id,split,field,metric,value,ground_truth_value,qwen_value,ground_truth_agreement_tier
0,Qwen3-VL-2B-Instruct,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,,on_screen_text,rouge,0.333333,"[Jacobin, on politics:]","[PROCLAMATION, JACOBIN, ISSUE 21, SPRING 2016,...",not_scored
1,Qwen3-VL-2B-Instruct,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,,visual_tags,accuracy,0.852941,"[reporter, interview, person speaking, text on...","[interview, person speaking, crowd of people, ...",not_scored
2,Qwen3-VL-2B-Instruct,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,,keywords,rouge,0.083333,"[ideas, publication, jackamon socialist, socia...","[Jacobin magazine, political ideology, sociali...",not_scored
3,Qwen3-VL-2B-Instruct,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,,people_count,accuracy,0.000000,10,15,not_scored
4,Qwen3-VL-2B-Instruct,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,,transcript,rouge,0.034483,Jackman is a socialist publication. We're rais...,The editor discusses the ideological roots of ...,not_scored
5,Qwen3-VL-2B-Instruct,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,,on_screen_text,rouge,0.000000,[Ronan Burtenshaw],"[REP, THE WORKS, REPU, 1916]",not_scored
6,Qwen3-VL-2B-Instruct,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,,visual_tags,accuracy,0.882353,"[audience, interview, reporter]","[interview, audience, stage, podium, graphic o...",not_scored
7,Qwen3-VL-2B-Instruct,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,,keywords,rouge,0.066667,"[debating, southern perspective, create broad,...","[political event, editor, Jacobin 1916, speech...",not_scored
8,Qwen3-VL-2B-Instruct,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,,people_count,accuracy,0.000000,4,7,not_scored
9,Qwen3-VL-2B-Instruct,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,,transcript,rouge,0.027027,and the things that they're debating and talki...,The speaker discusses the significance of Jaco...,not_scored


## 6. Comparison table: accuracy and rouge, per model, per field

In [ ]:
FIELD_METRIC = {
    "on_screen_text": "rouge",
    "keywords": "rouge",
    "visual_tags": "accuracy",
    "people_count": "accuracy",
    "transcript": "rouge",
}
FIELD_ORDER = ["on_screen_text", "keywords", "visual_tags", "people_count", "transcript"]
MODEL_ORDER = [m["name"] for m in MODEL_CONFIGS]

coverage_table = QWEN_ACCURACY_REPORT.pivot_table(
    index="model", columns="field", values="clip_id", aggfunc="nunique",
).reindex(index=MODEL_ORDER, columns=FIELD_ORDER).fillna(0).astype(int)
QWEN_COVERAGE_TABLE_FILE = CFG.qwen_output_dir / f"qwen_vl_coverage_table__{RUN_TAG}.csv"
write_csv_atomic(coverage_table.reset_index(), QWEN_COVERAGE_TABLE_FILE)
print(f"Clips scored per model x field (out of {len(CLIP_MANIFEST)} total):")
display(coverage_table)
print("Saved to:", QWEN_COVERAGE_TABLE_FILE)

comparison_table = QWEN_ACCURACY_REPORT.pivot_table(
    index="model", columns="field", values="value", aggfunc="mean",
).reindex(index=MODEL_ORDER, columns=FIELD_ORDER)
comparison_table.columns = [f"{field} ({FIELD_METRIC[field]})" for field in comparison_table.columns]
comparison_table = comparison_table.round(3)

QWEN_COMPARISON_TABLE_FILE = CFG.qwen_output_dir / f"qwen_vl_comparison_table__{RUN_TAG}.csv"
write_csv_atomic(comparison_table.reset_index(), QWEN_COMPARISON_TABLE_FILE)
print(f"\nMean score per model x field (n={len(CLIP_MANIFEST)} clips):")
display(comparison_table)
print("Saved to:", QWEN_COMPARISON_TABLE_FILE)


Clips scored per model x field (out of 615 total):


field,on_screen_text,keywords,visual_tags,people_count,transcript
model,,,,,
Qwen3-VL-2B-Instruct,615,615,615,560,615
Qwen3-VL-2B-Thinking,615,615,615,560,615
Qwen3-VL-4B-Instruct,615,615,615,560,615
Qwen3-VL-4B-Thinking,615,615,615,560,615
Qwen3-VL-8B-Instruct,615,615,615,560,615
InternVL3-2B,615,615,615,560,615
InternVL3-8B,615,615,615,560,615


Saved to: /content/drive/MyDrive/NVTV_PublicData1/automatic_ground_truth_v4/qwen_vl_comparison/qwen_vl_coverage_table__ALL_MODELS.csv

Mean score per model x field (n=615 clips):


,on_screen_text (rouge),keywords (rouge),visual_tags (accuracy),people_count (accuracy),transcript (rouge)
model,,,,,
Qwen3-VL-2B-Instruct,0.377,0.038,0.864,0.225,0.034
Qwen3-VL-2B-Thinking,0.382,0.028,0.876,0.321,0.018
Qwen3-VL-4B-Instruct,0.448,0.030,0.875,0.239,0.000
Qwen3-VL-4B-Thinking,0.386,0.040,0.867,0.307,0.018
Qwen3-VL-8B-Instruct,0.433,0.036,0.869,0.227,0.000
InternVL3-2B,0.566,0.030,0.895,0.404,0.000
InternVL3-8B,0.410,0.026,0.878,0.354,0.013


Saved to: /content/drive/MyDrive/NVTV_PublicData1/automatic_ground_truth_v4/qwen_vl_comparison/qwen_vl_comparison_table__ALL_MODELS.csv


## 7. Download the comparison files

In [ ]:
model_name = "Qwen3-VL-4B-Instruct"

transcript_rows = QWEN_ACCURACY_REPORT[
    (QWEN_ACCURACY_REPORT["model"] == model_name)
    & (QWEN_ACCURACY_REPORT["field"] == "transcript")
].copy()

transcript_rows["prediction_empty"] = (
    transcript_rows["qwen_value"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

print("Exact mean:", f"{transcript_rows['value'].mean():.8f}")
print("Rows scored:", len(transcript_rows))
print("Empty predictions:", transcript_rows["prediction_empty"].sum())
print("Non-zero scores:", (transcript_rows["value"] > 0).sum())
print("Maximum score:", transcript_rows["value"].max())

display(
    transcript_rows[
        ["clip_id", "ground_truth_value", "qwen_value", "value"]
    ].head(10)
)

Exact mean: 0.00014347
Rows scored: 615
Empty predictions: 613
Non-zero scores: 1
Maximum score: 0.08823529411764706


,clip_id,ground_truth_value,qwen_value,value
6044,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,Jackman is a socialist publication. We're rais...,,0.0
6049,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,and the things that they're debating and talki...,,0.0
6054,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,space for socialism in Ireland that talks abou...,,0.0
6059,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,change society you need to be able to talk to ...,,0.0
6064,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,Ireland actually also had a very weak left pos...,,0.0
6069,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,in the global south too. So Ireland's kind of ...,,0.0
6073,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,not just the rising. So it talks about the Lim...,,0.0
6077,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,States to some extent. And then the third part...,,0.0
6082,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,movement and that leads to some sort of tensio...,,0.0
6087,10_Jacobin_Launch_230316_13c6f149__030000ms__c...,of evolutionary period in our magazine. And we...,,0.0


In [ ]:
import zipfile

RESULTS_FILES = [
    QWEN_ACCURACY_REPORT_FILE,
    QWEN_COVERAGE_TABLE_FILE,
    QWEN_COMPARISON_TABLE_FILE,
]
RESULTS_FILES = [path for path in RESULTS_FILES if path.exists()]

RESULTS_ZIP_FILE = CFG.qwen_output_dir / f"qwen_vl_all_results__{RUN_TAG}.zip"
with zipfile.ZipFile(RESULTS_ZIP_FILE, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in RESULTS_FILES:
        archive.write(path, arcname=path.name)

print(f"Zipped {len(RESULTS_FILES)} file(s) into:")
print(f"  {RESULTS_ZIP_FILE}")
for path in RESULTS_FILES:
    print(f"    - {path.name}")

try:
    from google.colab import files as colab_files
except ImportError:
    print(f"\nNot running in Colab -- grab the file directly from: {RESULTS_ZIP_FILE}")
else:
    colab_files.download(str(RESULTS_ZIP_FILE))


Zipped 3 file(s) into:
  /content/drive/MyDrive/NVTV_PublicData1/automatic_ground_truth_v4/qwen_vl_comparison/qwen_vl_all_results__ALL_MODELS.zip
    - qwen_vl_accuracy_report__ALL_MODELS.csv
    - qwen_vl_coverage_table__ALL_MODELS.csv
    - qwen_vl_comparison_table__ALL_MODELS.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>